In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard imports
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import base64
import pandas as pd

JupyterDash.infer_jupyter_proxy_config()

# CRUD module import
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = 'REDACTED_COURSE_CREDENTIAL'
password = 'REDACTED_COURSE_CREDENTIAL'

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))

if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Logo
image_filename = "Grazioso Salvare Logo.png"

try:
    encoded_image = base64.b64encode(open(image_filename, "rb").read()).decode()
    logo = html.Img(
        src="data:image/png;base64,{}".format(encoded_image),
        style={"height": "120px"}
    )
except:
    logo = html.H3("Grazioso Salvare Logo")


app.layout = html.Div([
    html.Center([
        logo,
        html.H1("CS-340 Project Two Dashboard"),
        html.H2("Christopher Wright")
    ]),

    html.Hr(),

    html.Div([
        html.H3("Rescue Animal Filters"),

        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                {"label": "Reset", "value": "reset"}
            ],
            value="reset",
            labelStyle={"display": "inline-block", "margin-right": "20px"}
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        selected_columns=[],
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "200px",
            "whiteSpace": "normal"
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={"width": "50%"}
            ),
            html.Div(
                id="map-id",
                className="col s12 m6",
                style={"width": "50%"}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    if filter_type == "water":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "mountain":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "disaster":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    filtered_df = pd.DataFrame.from_records(db.read(query))

    if "_id" in filtered_df.columns:
        filtered_df.drop(columns=["_id"], inplace=True)

    return filtered_df.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame(viewData)

    if dff.empty or "breed" not in dff.columns:
        return [html.H3("No breed data available")]

    breed_count = dff["breed"].value_counts().nlargest(10).reset_index()
    breed_count.columns = ["breed", "count"]

    return [
        dcc.Graph(
            figure=px.pie(
                breed_count,
                names="breed",
                values="count",
                title="Top 10 Breeds in Current Filter"
            )
        )
    ]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[
                        dff.iloc[row]["location_lat"],
                        dff.iloc[row]["location_long"]
                    ],
                    children=[
                        dl.Tooltip(dff.iloc[row]["breed"]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row]["name"])
                        ])
                    ]
                )
            ]
        )
    ]


app.run_server(debug=True, port=8050)